# 11. Stable Diffusion 3 기반 이미지 생성

Kaggle Notebook에서 실행할 수 있는 Stable Diffusion 3 이미지 생성 예제입니다.

- `diffusers` 라이브러리 사용
- `StableDiffusion3Pipeline` 사용
- 텍스트 프롬프트를 입력으로 이미지 생성
- `num_inference_steps`, `guidance_scale`, `height`, `width` 옵션 포함
- 생성된 이미지를 출력하고 파일로 저장

> Stable Diffusion 3 모델은 Hugging Face에서 라이선스 동의와 토큰이 필요할 수 있습니다. Kaggle Notebook 오른쪽 설정에서 Internet을 켜고, 필요한 경우 Add-ons > Secrets에 `HF_TOKEN`을 등록하세요.

In [ ]:
# Kaggle 환경에서 필요한 패키지를 설치합니다.
# 설치 후 import 오류가 나면 Notebook을 Restart한 뒤 처음부터 다시 실행하세요.
!pip -q install -U diffusers transformers accelerate safetensors huggingface_hub sentencepiece protobuf

In [ ]:
from datetime import datetime
from pathlib import Path

import torch
from diffusers import StableDiffusion3Pipeline
from huggingface_hub import HfApi, login
from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
from IPython.display import display

# GPU가 있으면 cuda를 사용합니다. SD3는 GPU 사용을 강력히 권장합니다.
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

print("사용 장치:", device)
print("torch dtype:", torch_dtype)

## 1. Hugging Face 토큰 설정

Stable Diffusion 3 모델은 접근 권한이 필요할 수 있습니다. Kaggle Secrets에 `HF_TOKEN`을 넣어두면 아래 코드가 자동으로 로그인합니다.

1. Hugging Face에서 모델 페이지 라이선스에 동의합니다.
2. Hugging Face Access Token을 생성합니다.
3. Kaggle Notebook의 Add-ons > Secrets에 이름 `HF_TOKEN`으로 저장합니다.

In [ ]:
hf_token = None

try:
    from kaggle_secrets import UserSecretsClient

    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    login(token=hf_token)
    print("Hugging Face 로그인 완료")
else:
    print("HF_TOKEN이 없습니다. 모델 접근 오류가 나면 Kaggle Secrets에 HF_TOKEN을 추가하세요.")

# 토큰은 노트북 코드에 직접 적지 말고 Kaggle Secrets로 관리하세요.

## 2. Stable Diffusion 3 Pipeline 불러오기

In [ ]:
# 공식 diffusers 문서에서 사용하는 Stable Diffusion 3 Medium 모델입니다.
# 라이선스 동의가 필요할 수 있습니다.
model_id = "stabilityai/stable-diffusion-3-medium-diffusers"

# 모델 로딩 전에 접근 권한을 먼저 확인합니다.
# 여기서 401 또는 gated repo 오류가 나면 Hugging Face 모델 페이지에서 라이선스 동의가 필요합니다.
api = HfApi(token=hf_token)

try:
    api.model_info(model_id)
    print("모델 접근 권한 확인 완료:", model_id)
except GatedRepoError as error:
    raise RuntimeError(
        "Stable Diffusion 3 모델은 gated repo입니다. "
        "1) Hugging Face에서 모델 페이지 라이선스/접근 신청에 동의하고, "
        "2) 같은 계정의 Access Token을 Kaggle Secrets에 HF_TOKEN 이름으로 저장한 뒤, "
        "3) Notebook을 Restart하고 다시 실행하세요."
    ) from error
except HfHubHTTPError as error:
    raise RuntimeError(
        "Hugging Face 모델 접근 확인에 실패했습니다. "
        "HF_TOKEN이 없거나, 만료되었거나, 권한이 부족할 수 있습니다."
    ) from error

pipe = StableDiffusion3Pipeline.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    token=hf_token,
)

# SD3는 세 개의 텍스트 인코더를 사용해 메모리를 많이 씁니다.
# GPU 환경에서는 CPU offload를 켜서 Kaggle GPU 메모리 부담을 줄입니다.
if device == "cuda":
    pipe.enable_model_cpu_offload()
else:
    pipe = pipe.to(device)

print("Stable Diffusion 3 Pipeline 로딩 완료:", model_id)

## 3. 프롬프트와 생성 옵션 설정

SD3는 영어 프롬프트에서 가장 안정적으로 동작합니다. `height`와 `width`는 8의 배수로 설정하세요.

In [ ]:
# 만들고 싶은 이미지를 텍스트로 입력합니다.
prompt = "a cinematic photo of a glass greenhouse cafe in a rainy forest, soft warm lights, highly detailed"

# 원하지 않는 요소를 줄이기 위한 네거티브 프롬프트입니다.
negative_prompt = "low quality, blurry, distorted, deformed, text, watermark, logo"

# 생성 옵션입니다.
# num_inference_steps: 높을수록 세부 묘사가 좋아질 수 있지만 생성 시간이 늘어납니다.
# guidance_scale: 프롬프트를 얼마나 강하게 따를지 조절합니다.
# height, width: SD3는 1024x1024 품질이 좋지만 Kaggle GPU에서는 768부터 테스트해도 좋습니다.
num_inference_steps = 28
guidance_scale = 7.0
height = 768
width = 768

# 같은 이미지를 재현하고 싶으면 seed를 고정합니다.
seed = 42
generator = torch.Generator(device="cpu").manual_seed(seed)

print("Prompt:", prompt)
print("Negative prompt:", negative_prompt)
print("옵션:", num_inference_steps, guidance_scale, height, width, "seed=", seed)

## 4. 이미지 생성하기

In [ ]:
# StableDiffusion3Pipeline에 프롬프트와 옵션을 전달해 이미지를 생성합니다.
result = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=num_inference_steps,
    guidance_scale=guidance_scale,
    height=height,
    width=width,
    generator=generator,
)

image = result.images[0]
display(image)

## 5. 생성된 이미지 저장하기

In [ ]:
# Kaggle에서는 /kaggle/working 아래 파일이 Output으로 저장됩니다.
output_dir = Path("/kaggle/working/stable_diffusion3_outputs")

# 로컬 Jupyter에서 실행할 때 /kaggle/working이 없으면 현재 폴더에 저장합니다.
if not Path("/kaggle/working").exists():
    output_dir = Path("stable_diffusion3_outputs")

output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"stable_diffusion3_{timestamp}.png"

image.save(output_path)
print("저장 완료:", output_path)

## 6. 여러 프롬프트로 이미지 여러 장 생성하기

아래 셀은 여러 프롬프트를 반복 실행하고 각각 파일로 저장하는 예시입니다.

In [ ]:
prompts = [
    "a friendly robot barista making coffee in a tiny cafe, cinematic lighting",
    "a futuristic Korean city street at sunset, realistic photo, detailed architecture",
    "a magical library floating above the clouds, fantasy concept art, rich details",
]

generated_files = []

for index, prompt_text in enumerate(prompts, start=1):
    print(f"\n[{index}] 이미지 생성 중:", prompt_text)

    current_seed = seed + index
    current_generator = torch.Generator(device="cpu").manual_seed(current_seed)

    current_result = pipe(
        prompt=prompt_text,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        generator=current_generator,
    )

    current_image = current_result.images[0]
    display(current_image)

    current_path = output_dir / f"stable_diffusion3_{index:02d}_seed_{current_seed}.png"
    current_image.save(current_path)
    generated_files.append(str(current_path))

    print("저장 완료:", current_path)

generated_files